# Critical Input DEQN: Repair-Cost Threshold and Policy Tests

This notebook avoids rerunning the full baseline grid. It takes the no-relief repair-active anchor seriously, then moves only the repair-cost threshold and adds a compact policy comparison.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path('/content/econml')
if not (ROOT / 'src').exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))

from critical_input_sensitivity_helpers import train_rule_variant

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT_ROOT = ARTIFACT_ROOT / 'repair_threshold_policy_tests'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Medium bottleneck / no-relief environment. The cost dimension is the test object.
BASE_NO_RELIEF_MEDIUM = {
    'mark_D': 0.35,
    'nu_qD': 1.25,
    'nu_pX': 0.0,
    'nu_qX': 0.0,
    'beta_X': 0.0,
    'mark_X': 0.0,
    'log_bar_lambda_X': -30.0,
}

# 0.015 is normally loaded as an anchor from the existing interior-repair probes.
REPAIR_COSTS_TO_TRAIN = [0.020, 0.030]
POLICIES_FOR_THRESHOLD = ['fixed', 'bottleneck']

# Optional compact policy comparison at the repair-active cost.
TRAIN_REPAIR_AWARE_COST015 = True
REPAIR_AWARE_COST015_OVERRIDES = {
    'repair_cost_share_10pct': 0.015,
    # Keep default repair-aware rule coefficients unless you intentionally want an aggressive variant.
}

RUN_TRAIN = True
RETRAIN = False
TRAINING = dict(
    rule_steps=2500,
    qmc_train=128,
    qmc_val=256,
    n_val_states=512,
    stop_val_states=256,
    batch_size=1024,
    sim_batch_size=256,
    dtype='float64',
    natural_oracle_nodes=16,
    checkpoint_every=500,
    checkpoint_keep=6,
    scenario_q_weight=15.0,
    calm_anchor_weight=2.0,
    calm_residual_weight=2.0,
)

print('ROOT:', ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

for cost in REPAIR_COSTS_TO_TRAIN:
    name = f'cost_{cost:.3f}'.replace('.', 'p')
    train_rule_variant(
        root=ROOT,
        output_dir=OUT_ROOT / name,
        params={**BASE_NO_RELIEF_MEDIUM, 'repair_cost_share_10pct': float(cost)},
        policies=POLICIES_FOR_THRESHOLD,
        run_train=RUN_TRAIN,
        retrain=RETRAIN,
        **TRAINING,
    )

if TRAIN_REPAIR_AWARE_COST015:
    train_rule_variant(
        root=ROOT,
        output_dir=OUT_ROOT / 'cost_0p015_repair_aware',
        params={**BASE_NO_RELIEF_MEDIUM, **REPAIR_AWARE_COST015_OVERRIDES},
        policies=['repair_aware'],
        run_train=RUN_TRAIN,
        retrain=RETRAIN,
        **TRAINING,
    )


In [ ]:
from IPython.display import display
from critical_input_sensitivity_helpers import collect_rule_ir_diagnostics, plot_run_comparison, zip_folder

REPORT_DIR = OUT_ROOT / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ANCHOR_COST015 = {
    'fixed': ARTIFACT_ROOT / 'fixed_taylor_interior_repair_probe',
    'bottleneck': ARTIFACT_ROOT / 'bottleneck_interior_repair_probe',
    'repair_aware': ARTIFACT_ROOT / 'repair_aware_interior_repair_probe',
}

runs = []
for policy, folder in ANCHOR_COST015.items():
    if (folder / 'run_config.json').exists() and ((folder / 'checkpoints' / f'{policy}_best.pt').exists() or (folder / f'{policy}.pt').exists()):
        runs.append({'variant': 'cost_0p015_anchor', 'policy': policy, 'output_dir': folder})
    else:
        print('Skipping missing cost-0.015 anchor:', policy, folder)

if (OUT_ROOT / 'cost_0p015_repair_aware' / 'run_config.json').exists():
    runs.append({'variant': 'cost_0p015_trained', 'policy': 'repair_aware', 'output_dir': OUT_ROOT / 'cost_0p015_repair_aware'})

for cost in REPAIR_COSTS_TO_TRAIN:
    variant = f'cost_{cost:.3f}'.replace('.', 'p')
    for policy in POLICIES_FOR_THRESHOLD:
        runs.append({'variant': variant, 'policy': policy, 'output_dir': OUT_ROOT / variant})

detail, summary, series_by_run = collect_rule_ir_diagnostics(
    artifact_root=ARTIFACT_ROOT,
    runs=runs,
    report_dir=REPORT_DIR,
    dtype=TRAINING['dtype'],
    burnin=80,
    horizon=100,
    presteps=5,
    relief_lag=8,
    natural_oracle_nodes=16,
    plot_each=False,
)

print('\nKEY SCENARIO SUMMARY')
display(summary)

for scenario in ['D_1x', 'D_3x', 'D_3x_X_lag']:
    plot_run_comparison(
        series_by_run,
        report_dir=REPORT_DIR,
        scenario=scenario,
        filename=f'{scenario}_repair_threshold_policy_comparison.png',
    )

ZIP_PATH = Path('/content/repair_threshold_policy_tests.zip')
zip_folder(OUT_ROOT, ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print('Download helper skipped:', exc)
